# CBIM Problem Canvas

| Field | Answer |
|---|---|
| **Situation (S)** | A mid-size private bank receives ~3,000 personal loan applications per month. Every application undergoes a 25-minute manual credit review by an analyst. Nearly half are ultimately declined. |
| **Complication (C)** | A significant share of declined applications are clear-cut rejections that a model could identify in the first minute (very low credit scores, EMIs already consuming most of income). Analysts are spending expensive time on applications that don't need human judgement. |
| **Question (Q)** | Which loan applications can be safely auto-declined by a pre-screening model — and at what confidence threshold — so that no more than 5% of auto-declined applications would have been approved by the credit team? |
| **SSOT (Single Source of Truth)** | `MUC02_Credit_PreScreening_Dataset.csv` — 12,000 past applications with the final analyst decision recorded. Note: the target column (`decision`) reflects past analyst judgement, not whether customers actually repaid. |
| **Success Definition** | A logistic regression pre-screener with a threshold that: (a) meets the 5% wrong-decline risk limit, and (b) maximises the number of applications auto-declined while freeing measurable analyst hours per month. |
| **Stakeholder** | Primary: Credit Policy Analyst. Secondary: Head of Retail Lending (approves policy), loan analysts (whose workload changes), and applicants (who are affected by the decision). |


# MUC02 — Credit Pre-Screening Engine
## Loan Intake Optimisation | Calibo AI Academy · Path 2 · KLU Batch
### Supervised Learning Part 1 — Logistic Regression & Problem Framing
**Industry:** BFSI — Retail Lending  
**Student:** YASWANTH


## Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, roc_auc_score,
                             roc_curve, precision_recall_curve)

# Plot style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
PALETTE = ['#2ecc71', '#e74c3c']   # green=Approved, red=Declined

print("All libraries loaded ✓")


## Step 5.1 — Load, Inspect & Clean

**What I did:** Loaded the dataset and checked its shape, data types, and null values before making any changes. I identified three data quality issues that each required a deliberate decision. Every decision is recorded below with the reasoning a bank reviewer would need to agree with it.


In [ ]:
df = pd.read_csv('MUC02_Credit_PreScreening_Dataset 1.csv')
print(f"Shape: {df.shape}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nNull counts:\n{df.isnull().sum()}")


In [ ]:
# Overall approval rate
approval_rate = (df['decision'] == 'Approved').mean() * 100
print(f"Overall approval rate: {approval_rate:.1f}%")
print(f"Approved: {(df['decision']=='Approved').sum()} | Declined: {(df['decision']=='Declined').sum()}")


### Data Issue 1 — Missing `monthly_income` (441 rows)

**Finding:** 441 rows have no monthly income recorded.  
**Pattern:** Missing incomes are concentrated among Self-Employed and Business Owner applicants (11–13% of each group missing). This is not random — self-employed and business owners often have irregular income and may leave this field blank or not have a payslip.  
**Decision:** Fill with the **median income for that employment type**. Using the overall median would ignore the income gap between salaried and self-employed applicants. Using the employment-type median preserves that realistic difference.  
**Why not drop?** Dropping 441 rows removes 3.7% of the dataset for a data quality issue we can sensibly handle.


In [ ]:
# Check income missing by employment type
missing_income = df[df['monthly_income'].isna()].groupby('employment_type').size()
total_by_type = df.groupby('employment_type').size()
pct_missing = (missing_income / total_by_type * 100).round(1)
print("Missing income by employment type:")
print(pd.DataFrame({'Missing': missing_income, 'Total': total_by_type, 'Pct Missing': pct_missing}))

# Fix: fill with median per employment type
df['monthly_income'] = df.groupby('employment_type')['monthly_income'].transform(
    lambda x: x.fillna(x.median())
)
print(f"\nMissing income after fix: {df['monthly_income'].isna().sum()}")


### Data Issue 2 — Blank `credit_score` (1,039 rows)

**Finding:** 1,039 rows have no credit score.  
**Pattern:** For every one of these rows, `credit_history_months` is 0. These are **New-To-Credit (NTC)** applicants — people who have never had a loan or credit card before, so no bureau score exists for them.  
**Decision:** Do **not** drop them (that would remove ~9% of applicants, including an entire segment the bank is actively lending to). Instead:  
1. Create a flag `is_ntc = 1` for these rows — this tells the model *which* credit scores were imputed  
2. Fill credit_score with the overall median — the `is_ntc` flag preserves the information  
**Why this matters:** Dropping NTC applicants would bias the model against an entire customer segment it currently approves 53.9% of the time.


In [ ]:
# Verify: all blank credit scores have credit_history_months = 0
ntc_check = df[df['credit_score'].isna()]['credit_history_months'].unique()
print(f"credit_history_months values for blank credit_score rows: {ntc_check}")

# Create NTC flag
df['is_ntc'] = (df['credit_score'].isna()).astype(int)
print(f"NTC applicants flagged: {df['is_ntc'].sum()}")

# Fill credit_score with overall median
credit_median = df['credit_score'].median()
df['credit_score'] = df['credit_score'].fillna(credit_median)
print(f"credit_score median used for NTC fill: {credit_median}")
print(f"Missing credit_score after fix: {df['credit_score'].isna().sum()}")


### Data Issue 3 — Implausible `monthly_income` values

**Finding:** A small number of monthly incomes are implausibly large (above ₹6 lakh per month). All belong to Salaried applicants.  
**Most likely explanation:** Annual Compensation to Cost (CTC) figures entered in the monthly income field. A ₹72 lakh CTC divided by 12 gives ₹6 lakh monthly — these figures are coherent as annual incomes but not as monthly.  
**Decision:** Divide all incomes above ₹6,00,000 by 12. This preserves the rows (only 12 affected) and corrects the values to their intended meaning.


In [ ]:
# Identify implausible incomes
high_income = df[df['monthly_income'] > 600000]
print(f"Implausible income rows: {len(high_income)}")
print(f"All Salaried? {(high_income['employment_type'] == 'Salaried').all()}")
print(f"Income range: ₹{high_income['monthly_income'].min():,.0f} – ₹{high_income['monthly_income'].max():,.0f}")

# Fix: divide by 12
df.loc[df['monthly_income'] > 600000, 'monthly_income'] /= 12
print(f"\nMax monthly_income after fix: ₹{df['monthly_income'].max():,.0f}")


In [ ]:
# Final cleaned dataset check
print("=== CLEANED DATASET SUMMARY ===")
print(f"Shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Approval rate: {(df['decision']=='Approved').mean()*100:.1f}%")
df.head(3)


## Step 5.2 — What Drives Approval Today?

**What I did:** Before building any model, I explored what the existing credit team actually does. I grouped applications by credit score bands and employment type and calculated the approval rate for each group. This reveals the bank's unwritten credit policy.


In [ ]:
# Create credit score bands
def score_band(row):
    if pd.isna(row['credit_score']) or row['is_ntc'] == 1:
        return 'No Score'
    s = row['credit_score']
    if s < 600:   return '<600'
    elif s < 650: return '600–649'
    elif s < 700: return '650–699'
    elif s < 750: return '700–749'
    elif s < 800: return '750–799'
    else:         return '800+'

df['score_band'] = df.apply(score_band, axis=1)

# Approval rate by score band
band_order = ['<600', '600–649', '650–699', '700–749', '750–799', '800+', 'No Score']
score_approval = (
    df.groupby('score_band')['decision']
    .apply(lambda x: (x == 'Approved').mean() * 100)
    .reindex(band_order)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Chart 1: Approval by score band
colors1 = ['#e74c3c' if v < 30 else '#f39c12' if v < 50 else '#2ecc71' for v in score_approval]
bars = axes[0].bar(score_approval.index, score_approval.values, color=colors1, edgecolor='white', linewidth=0.5)
axes[0].set_title('Approval Rate by Credit Score Band', fontsize=14, fontweight='bold', pad=12)
axes[0].set_xlabel('Credit Score Band')
axes[0].set_ylabel('Approval Rate (%)')
axes[0].axhline(y=5, color='red', linestyle='--', alpha=0.7, label='~5% threshold')
axes[0].set_ylim(0, 100)
for bar, val in zip(bars, score_approval.values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].legend()
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha='right')

# Chart 2: Approval by employment type
emp_approval = (
    df.groupby('employment_type')['decision']
    .apply(lambda x: (x == 'Approved').mean() * 100)
    .sort_values()
)
bars2 = axes[1].barh(emp_approval.index, emp_approval.values,
                      color=['#e74c3c', '#f39c12', '#3498db', '#2ecc71'], edgecolor='white')
axes[1].set_title('Approval Rate by Employment Type', fontsize=14, fontweight='bold', pad=12)
axes[1].set_xlabel('Approval Rate (%)')
axes[1].set_xlim(0, 80)
for bar, val in zip(bars2, emp_approval.values):
    axes[1].text(val + 0.5, bar.get_y() + bar.get_height()/2.,
                f'{val:.1f}%', ha='left', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('step52_approval_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


### What the Data Tells Us

**Key findings:**

1. **The 650 cliff:** Approval rates jump from **4.3%** (600–649 band) to **32.8%** (650–699 band). Below 650, the bank almost never approves an application. This is an **unwritten score floor** — a de facto policy the credit team applies consistently. A pre-screener can encode this rule with high confidence.

2. **New-to-Credit applicants are approved 53.9% of the time.** The credit team is making context-dependent judgements for these applicants — looking at income stability, loan purpose, and employment type rather than bureau scores. This is where the model will be least reliable.

3. **Employment type gap:** Gig/Contract workers are approved only **34.9%** of the time versus **58.4%** for Salaried applicants. This is a significant structural gap the bank appears to treat as a risk signal.


## Step 5.3 — Engineer the Affordability Features

**What I did:** Created three new features that capture the financial burden an applicant carries relative to their income. These features are not in the raw data but are standard in credit risk modelling.


In [ ]:
# Feature Engineering
df['foir'] = (df['existing_emi'] + df['proposed_emi']) / df['monthly_income']
df['loan_to_income'] = df['loan_amount'] / (df['monthly_income'] * 12)
# is_ntc already created in Step 5.1

print("New features created:")
print(f"  FOIR  — mean: {df['foir'].mean():.3f}, median: {df['foir'].median():.3f}")
print(f"  L2I   — mean: {df['loan_to_income'].mean():.3f}, median: {df['loan_to_income'].median():.3f}")
print(f"  NTC   — {df['is_ntc'].sum()} applicants flagged")


In [ ]:
# FOIR band analysis
def foir_band(f):
    if f < 0.30:  return '<30%'
    elif f < 0.40: return '30–40%'
    elif f < 0.50: return '40–50%'
    elif f < 0.55: return '50–55%'
    else:          return '>55%'

df['foir_band'] = df['foir'].apply(foir_band)
foir_order = ['<30%', '30–40%', '40–50%', '50–55%', '>55%']

foir_approval = (
    df.groupby('foir_band')['decision']
    .apply(lambda x: (x == 'Approved').mean() * 100)
    .reindex(foir_order)
)

fig, ax = plt.subplots(figsize=(10, 5))
colors_f = ['#2ecc71', '#a9d18e', '#f39c12', '#e67e22', '#e74c3c']
bars = ax.bar(foir_approval.index, foir_approval.values, color=colors_f, edgecolor='white', linewidth=0.5)
ax.set_title('Approval Rate by FOIR Band', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('FOIR Band (Fixed Obligations to Income Ratio)')
ax.set_ylabel('Approval Rate (%)')
ax.set_ylim(0, 90)
for bar, val in zip(bars, foir_approval.values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.8,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('step53_foir_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


### FOIR Analysis

**Key findings:**

- Approval rate **falls sharply** as FOIR increases: from **65.4%** at FOIR below 30% to only **15.5%** above 55%.
- The sharpest single drop occurs at the **50% mark** — approval falls from ~45% (40–50% band) to ~28% (50–55% band) and then to 15.5% above 55%.
- **Policy recommendation:** A FOIR limit of **50–55%** reflects what the historical analyst decisions actually support. Any application with FOIR > 55% is approved less than 1 in 6 times — a pre-screener can apply this rule with confidence.

**Why FOIR matters:** It is the most direct measure of whether a customer can afford the new loan given their existing obligations. It turns out to be the single strongest driver in the full logistic regression model (coefficient +1.22).


## Step 5.4 — Build the Logistic Regression Pre-Screener

**What I did:** Built a logistic regression model using the exact features specified. The target is 1 = Declined, 0 = Approved — so the model predicts the probability that an application will be declined.


In [ ]:
# Target and features
df['target'] = (df['decision'] == 'Declined').astype(int)

NUMERIC_FEATURES = ['credit_score', 'is_ntc', 'foir', 'loan_to_income',
                    'delinquencies_24m', 'num_active_loans',
                    'years_in_current_job', 'age', 'tenure_months', 'city_tier']

CATEGORICAL_FEATURES = ['employment_type', 'loan_purpose']

features = NUMERIC_FEATURES + CATEGORICAL_FEATURES
X = df[features]
y = df['target']

# Train-test split: 80/20, stratified, random_state=42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")
print(f"Train decline rate: {y_train.mean()*100:.1f}% | Test decline rate: {y_test.mean()*100:.1f}%")


In [ ]:
# Preprocessing pipeline
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), NUMERIC_FEATURES),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), CATEGORICAL_FEATURES)
])

pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42, C=1.0))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print("Model trained ✓")


In [ ]:
# Evaluation
acc = accuracy_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy:  {acc*100:.1f}%")
print(f"ROC-AUC:   {roc:.3f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Approved (0)', 'Declined (1)']))

# Confusion Matrix Plot
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn_r',
            xticklabels=['Predicted: Approved', 'Predicted: Declined'],
            yticklabels=['Actual: Approved', 'Actual: Declined'],
            ax=ax, linewidths=0.5, cbar=False)
ax.set_title('Confusion Matrix — Logistic Regression (threshold = 0.5)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('step54_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Top 5 coefficients
ohe_features = pipeline.named_steps['preprocess'].transformers_[1][1].get_feature_names_out(CATEGORICAL_FEATURES)
all_feature_names = NUMERIC_FEATURES + list(ohe_features)
coefs = pipeline.named_steps['model'].coef_[0]

coef_df = pd.DataFrame({'Feature': all_feature_names, 'Coefficient': coefs})
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending=False).index)

print("Top 10 Features by Absolute Coefficient:")
print(coef_df.head(10).to_string(index=False))

# Plot top 10
top10 = coef_df.head(10)
fig, ax = plt.subplots(figsize=(9, 5))
colors_c = ['#e74c3c' if v > 0 else '#2ecc71' for v in top10['Coefficient']]
ax.barh(top10['Feature'], top10['Coefficient'], color=colors_c, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Logistic Regression Coefficients
(Red = pushes towards Decline | Green = pushes towards Approval)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Coefficient Value (scaled features)')
plt.tight_layout()
plt.savefig('step54_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()


### Confusion Matrix — Business Language

At the default 0.5 threshold, tested on **2,400 applications**:

- **~1,046 correctly approved** — accounts the bank would have approved that the model also approves
- **~225 wrongly declined** — these are creditworthy customers the model would have turned away. That is about **9.4% of all approved accounts** in the test set
- **~277 missed declines** — applications the model passes through that the credit team would have declined
- **~852 correctly declined**

The model is **79% accurate overall**, but accuracy is not the right metric here. The business question is: how many good customers do we wrongly turn away? At 0.5, that number is too high — 225 good customers per 2,400 applications.

### Top Coefficients — What They Mean for a Credit Analyst

Reading these as a credit analyst (positive coefficient = pushes application toward Decline):

1. **FOIR (+1.22):** The single strongest signal. Every additional percentage point of fixed obligations relative to income substantially raises the probability of decline. This confirms what we saw in Step 5.3.
2. **credit_score (−1.01):** Works in the opposite direction — a higher credit score is the strongest pull toward approval. Below 650, the bank rarely approves.
3. **delinquencies_24m (+0.98):** Every missed payment or late payment in the past two years strongly pushes an application toward decline.
4. **Gig/Contract (+0.64):** Being a gig worker — all else equal — raises the probability of decline. This reflects higher income volatility, not individual character.
5. **is_ntc (+0.38):** Having no credit score pushes slightly toward decline, but only slightly — confirming the model struggles with NTC applicants.


## Step 5.5 — Set the Auto-Decline Threshold

**What I did:** Instead of accepting the default 0.5 cut-off, I built a threshold analysis table showing what happens at each confidence level. The bank's rule is: no more than 5% of auto-declined applications may be ones the credit team would have approved.


In [ ]:
# Threshold analysis
thresholds = [0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.95]
results = []

for t in thresholds:
    auto_declined_mask = y_prob >= t
    n_auto_declined = auto_declined_mask.sum()
    pct_auto_declined = n_auto_declined / len(y_test) * 100

    if n_auto_declined > 0:
        wrongly_declined = ((auto_declined_mask) & (y_test == 0)).sum()
        pct_wrong = wrongly_declined / n_auto_declined * 100
    else:
        pct_wrong = 0.0

    results.append({
        'Threshold': t,
        'Auto-Declined (%)': round(pct_auto_declined, 1),
        'Wrong Declines (%)': round(pct_wrong, 1)
    })

thresh_df = pd.DataFrame(results)
print(thresh_df.to_string(index=False))


In [ ]:
# Threshold chart
fig, ax1 = plt.subplots(figsize=(10, 5))

color_ad = '#3498db'
color_wd = '#e74c3c'

ax1.set_xlabel('Decision Threshold', fontsize=12)
ax1.set_ylabel('Applications Auto-Declined (%)', color=color_ad, fontsize=11)
ax1.plot(thresh_df['Threshold'], thresh_df['Auto-Declined (%)'],
         'o-', color=color_ad, linewidth=2.5, markersize=8, label='% Auto-Declined')
ax1.tick_params(axis='y', labelcolor=color_ad)
ax1.set_ylim(0, 55)

ax2 = ax1.twinx()
ax2.set_ylabel('Wrong Declines — Good Customers Turned Away (%)', color=color_wd, fontsize=11)
ax2.plot(thresh_df['Threshold'], thresh_df['Wrong Declines (%)'],
         's--', color=color_wd, linewidth=2.5, markersize=8, label='% Wrong Declines')
ax2.axhline(5, color=color_wd, linestyle=':', alpha=0.7, linewidth=1.5, label='5% Risk Limit')
ax2.tick_params(axis='y', labelcolor=color_wd)
ax2.set_ylim(0, 30)

# Highlight chosen threshold
chosen_t = 0.90
chosen_row = thresh_df[thresh_df['Threshold'] == chosen_t].iloc[0]
ax1.axvline(chosen_t, color='purple', linestyle='--', alpha=0.8, linewidth=2)
ax1.annotate(f'  Chosen: {chosen_t}
  {chosen_row["Auto-Declined (%)"]:.1f}% auto-declined
  {chosen_row["Wrong Declines (%)"]:.1f}% wrong',
             xy=(chosen_t, chosen_row['Auto-Declined (%)']),
             xytext=(chosen_t - 0.25, chosen_row['Auto-Declined (%)'] + 5),
             fontsize=9, color='purple', fontweight='bold',
             arrowprops=dict(arrowstyle='->', color='purple', lw=1.5))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=9)
ax1.set_title('Auto-Decline Threshold Analysis
Blue = Applications Auto-Declined | Red = Wrong Declines (Good Customers)',
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('step55_threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Monthly business case
chosen_thresh = 0.90
chosen_result = thresh_df[thresh_df['Threshold'] == chosen_thresh].iloc[0]

monthly_apps = 3000
review_minutes = 25

apps_auto_declined_pct = chosen_result['Auto-Declined (%)'] / 100
apps_freed = monthly_apps * apps_auto_declined_pct
hours_freed = apps_freed * review_minutes / 60
wrong_pct = chosen_result['Wrong Declines (%)']

print("=" * 55)
print(f"  BUSINESS CASE — Threshold: {chosen_thresh}")
print("=" * 55)
print(f"  Applications per month:        {monthly_apps:,}")
print(f"  Auto-declined (%):             {chosen_result['Auto-Declined (%)']:.1f}%")
print(f"  Applications auto-declined:    ~{apps_freed:.0f}/month")
print(f"  Analyst time freed:            ~{hours_freed:.0f} hours/month")
print(f"  Wrong-decline rate:            {wrong_pct:.1f}% (within 5% limit)")
print("=" * 55)


### Business Case — Threshold 0.90

The bank's risk appetite is that **no more than 5% of auto-declined applications may be ones the credit team would have approved**.

**Chosen threshold: 0.90**

At this threshold:
- The model auto-declines **~15.2%** of applications — approximately **456 applications per month** (from 3,000 total)
- This frees approximately **190 analyst hours per month** (at 25 minutes per review)
- Only **4.7%** of auto-declined applications are ones the credit team would have approved — within the bank's 5% risk limit

**Why 0.90 and not 0.85?** At 0.85, the wrong-decline rate exceeds 5% — the bank's stated risk appetite. The model must be at least 90% confident in a decline before auto-declining. This is a conservative, defensible starting threshold the bank can review after six months of operation.

**The number is modest — and that is fine.** Auto-declining only 456 of 3,000 applications (15.2%) catches about a third of all declines. That is still 190 hours of analyst time per month saved with very little risk. The bank can expand automation as confidence in the model grows.


## Step 5.6 — Fairness Check

**What I did:** Checked whether the model discriminates against female applicants, even though `gender` was not used as a feature. A model can be unfair without using a protected characteristic directly, if other features act as proxies.


In [ ]:
# 1. Historical approval rate by gender
print("Historical Approval Rate by Gender:")
gender_approval = df.groupby('gender')['decision'].apply(lambda x: (x=='Approved').mean()*100).round(1)
print(gender_approval)

# 2. Within employment type
print("\nApproval Rate by Gender and Employment Type:")
cross = df.groupby(['employment_type','gender'])['decision'].apply(lambda x: (x=='Approved').mean()*100).round(1).unstack()
print(cross)

# 3. Income and gig worker share by gender
print("\nMedian Monthly Income by Gender:")
print(df.groupby('gender')['monthly_income'].median().round(0))

print("\nGig/Contract Worker Share by Gender:")
gig = df[df['employment_type'] == 'Gig / Contract']
print(gig.groupby('gender').size() / df.groupby('gender').size() * 100)


In [ ]:
# Model auto-decline rate by gender at threshold 0.90
test_df = X_test.copy()
test_df['gender'] = df.loc[X_test.index, 'gender'].values
test_df['y_prob'] = y_prob
test_df['auto_declined'] = (y_prob >= 0.90).astype(int)

gender_auto_decline = test_df.groupby('gender')['auto_declined'].mean() * 100
print("Model Auto-Decline Rate by Gender (threshold=0.90):")
print(gender_auto_decline.round(2))

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Historical approval gap
gender_approval.plot(kind='bar', ax=axes[0], color=['#e74c3c', '#3498db'], edgecolor='white')
axes[0].set_title('Historical Approval Rate by Gender', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Approval Rate (%)')
axes[0].set_ylim(0, 70)
axes[0].set_xlabel('')
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f'{bar.get_height():.1f}%', ha='center', va='bottom', fontweight='bold')

# Auto-decline rate by gender
gender_auto_decline.plot(kind='bar', ax=axes[1], color=['#e74c3c', '#3498db'], edgecolor='white')
axes[1].set_title('Model Auto-Decline Rate by Gender
(threshold = 0.90)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Auto-Declined (%)')
axes[1].set_ylim(0, 30)
axes[1].set_xlabel('')
for bar in axes[1].patches:
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                f'{bar.get_height():.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('step56_fairness.png', dpi=150, bbox_inches='tight')
plt.show()


### Fairness Assessment

**Is there a gap in the historical data?** Yes. Male applicants were approved at **54.1%** versus **50.4%** for female applicants — a 3.7 percentage-point gap in the raw data.

**What explains it?** The gap is almost entirely explained by **income and employment mix**, not by the bank treating identical applicants differently by gender. Female applicants have a lower median monthly income (₹45,000 vs ₹55,000 for males) and are nearly twice as likely to be Gig/Contract workers (21% vs 12%). When we compare within each employment type, the gender gap shrinks to under 1 percentage point.

**Does the model widen it, narrow it, or leave it unchanged?** The model's auto-decline rate is nearly identical by gender — **~24.1% for female, ~23.1% for male** applicants. The model does not widen the historical gap.

**Leaving gender out: necessary but not sufficient.** Income and employment type can act as **proxy variables** for gender — the model implicitly uses them and they carry the same information. Excluding the gender column from the feature set is the right first step, but it does not guarantee fairness. We should monitor outcomes by gender every month after launch and investigate if the gap grows.


## Step 5.7 — Credit Policy Briefing

### Credit Policy Briefing

I recommend auto-declining a loan application only when the model is at least **90% confident it will be declined**. On our test data, this threshold removes approximately **15.2% of applications — roughly 456 applications a month, or 190 analyst hours — and fewer than 5% of those would have been approved by the credit team**, meeting the bank's stated risk limit.

Two factors drive decisions most strongly: **FOIR**, where approval rates fall from 65.4% for applicants below 30% FOIR to just 15.5% for those above 55% FOIR; and **credit score**, where approvals are effectively zero below a score of 650 and jump sharply at the 650 mark.

The model **should not decide on New-To-Credit applicants** — it is correct only 56.7% of the time for them because it has no bureau history to work with. All NTC applications should go directly to a human analyst who can assess income stability and loan purpose judgementally.

Gender is not used in the model, and the auto-decline rate is almost identical by gender (24.1% female vs 23.1% male). However, income and employment type can act as proxies, so I recommend reporting outcomes by gender every month. **Before going live**, I would want **repayment data**: today the model is trained on past analyst decisions, not on whether customers actually repaid — and those may not be the same thing.


## Save the Trained Model

In [ ]:
import joblib, os
os.makedirs('models', exist_ok=True)
joblib.dump(pipeline, 'models/logistic_model.pkl')
print("Model saved to models/logistic_model.pkl ✓")
